<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [7]:
# My baseline prioritizes pages that are both stale and visibly active in search.
# I use days_since_last_update as the staleness signal because staleness is
# directly linked to FlyRank's documented refresh logic.
# I use impressions_90d as the visibility signal because meaningful search
# exposure indicates that a page has observable search visibility.
# The rule will assign one reason code, stale_visible_page, when both
# staleness and meaningful visibility are present.
# The corresponding action label will be REVIEW_REFRESH.
# The baseline is intended for decision-support and does not claim that
# refreshing a page will cause its performance to improve.

# ============================================================
# ML-07 — PART 1
# MY RULE AND ITS REASON CODES
# ============================================================

%pip install -q pandas numpy

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# LOAD THE STARTER DATASET
# ------------------------------------------------------------

possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv"
]

csv_path = next(
    (p for p in possible_paths if os.path.exists(p)),
    None
)

# If the repo file is not mounted, download the public starter CSV
# from the official FlyRank starter repository.
if csv_path is None:
    csv_url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    csv_path = "/content/content_refresh_anonymized.csv"
    df = pd.read_csv(csv_url)
else:
    df = pd.read_csv(csv_path)

print("Dataset loaded:", df.shape)

# ------------------------------------------------------------
# CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "days_since_last_update",
    "content_age_days"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# RULE
# ------------------------------------------------------------

print("""
BASELINE RULE

I will prioritize pages for refresh review when they are both:

1. Stale: at least 180 days since the last update.
2. Visible: at least 500 search impressions in the 90-day window.

The score will combine staleness and search visibility so that
older and more visible pages rank higher.

The rule does not use the decline label or any future-window data.
""")

# ------------------------------------------------------------
# SIGNAL CHECK 1 — STALENESS
# ------------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, np.inf],
    labels=["0-30", "31-90", "91-180", "180+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .size()
      .reset_index(name="n")
)

print("=" * 60)
print("SIGNAL CHECK 1 — STALENESS")
print("=" * 60)
display(staleness_table)

print("""
VERDICT: CONFIRMED

Staleness is a directly relevant refresh signal because FlyRank's
documented refresh logic includes pages that have not been updated
for a long time. The 180+ day bucket is therefore a reasonable
baseline threshold to investigate.
""")

# ------------------------------------------------------------
# SIGNAL CHECK 2 — SEARCH VISIBILITY / VOLUME
# ------------------------------------------------------------

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[-np.inf, 0, 100, 500, 3000, 30000, np.inf],
    labels=[
        "none",
        "1-100",
        "101-500",
        "501-3000",
        "3001-30000",
        "30000+"
    ]
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .size()
      .reset_index(name="n")
)

print("=" * 60)
print("SIGNAL CHECK 2 — SEARCH VISIBILITY")
print("=" * 60)
display(visibility_table)

print("""
VERDICT: CONFIRMED

Search impressions are a useful visibility signal because a page
with meaningful search exposure has more observable opportunity
for a refresh review than a page with no search exposure.
""")

print("\nRule reason code:")
print("stale_visible_page")

print("\nAction label:")
print("REVIEW_REFRESH")


Dataset loaded: (30000, 44)

BASELINE RULE

I will prioritize pages for refresh review when they are both:

1. Stale: at least 180 days since the last update.
2. Visible: at least 500 search impressions in the 90-day window.

The score will combine staleness and search visibility so that
older and more visible pages rank higher.

The rule does not use the decline label or any future-window data.

SIGNAL CHECK 1 — STALENESS


,staleness_bucket,n
0,0-30,20480
1,31-90,175
2,91-180,9171
3,180+,174



VERDICT: CONFIRMED

Staleness is a directly relevant refresh signal because FlyRank's
documented refresh logic includes pages that have not been updated
for a long time. The 180+ day bucket is therefore a reasonable
baseline threshold to investigate.

SIGNAL CHECK 2 — SEARCH VISIBILITY


,visibility_bucket,n
0,none,0
1,1-100,8006
2,101-500,5279
3,501-3000,8432
4,3001-30000,7205
5,30000+,1078



VERDICT: CONFIRMED

Search impressions are a useful visibility signal because a page
with meaningful search exposure has more observable opportunity
for a refresh review than a page with no search exposure.


Rule reason code:
stale_visible_page

Action label:
REVIEW_REFRESH


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# I use a simple transparent score so that every page can be ranked
# using the same rule.
# The score combines staleness and search visibility.
# Older pages receive more staleness points, while pages with higher
# impressions receive more visibility points.
# The final score is used only to prioritize pages for human review.
# A single reason code, stale_visible_page, identifies the main reason
# for the refresh recommendation.
# The action label REVIEW_REFRESH tells the reviewer what to do next.
# The ranked queue is written to work/outputs/baseline_action_score.csv.
# The rule does not use trend_direction, trend_pct, is_declining_label,
# FlyRank product flags, or future-window information.

# ============================================================
# ML-07 — PART 2
# BUILD THE RANKED QUEUE
# ============================================================

%pip install -q pandas numpy

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

possible_paths = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "/content/data/raw/content_refresh_anonymized.csv"
]

csv_path = next(
    (p for p in possible_paths if os.path.exists(p)),
    None
)

if csv_path is None:
    csv_url = (
        "https://raw.githubusercontent.com/"
        "flyrank-bih/flyrank-ml-internship-starter/"
        "main/data/raw/content_refresh_anonymized.csv"
    )
    csv_path = "/content/content_refresh_anonymized.csv"
    df = pd.read_csv(csv_url)
else:
    df = pd.read_csv(csv_path)

# ------------------------------------------------------------
# VALIDATE REQUIRED COLUMNS
# ------------------------------------------------------------

required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "days_since_last_update"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# CLEAN NUMERIC FIELDS
# ------------------------------------------------------------

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
).fillna(0)

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
).fillna(0)

# ------------------------------------------------------------
# SCORE COMPONENT 1 — STALENESS
# ------------------------------------------------------------

# 0 points below 90 days
# 1 point from 90 to <180 days
# 2 points at 180+ days

df["staleness_score"] = np.select(
    [
        df["days_since_last_update"] >= 180,
        df["days_since_last_update"] >= 90
    ],
    [
        2,
        1
    ],
    default=0
)

# ------------------------------------------------------------
# SCORE COMPONENT 2 — VISIBILITY
# ------------------------------------------------------------

# 0 = no meaningful visibility
# 1 = 100+ impressions
# 2 = 500+ impressions
# 3 = 3000+ impressions

df["visibility_score"] = np.select(
    [
        df["impressions_90d"] >= 3000,
        df["impressions_90d"] >= 500,
        df["impressions_90d"] >= 100
    ],
    [
        3,
        2,
        1
    ],
    default=0
)

# ------------------------------------------------------------
# FINAL BASELINE SCORE
# ------------------------------------------------------------

df["baseline_score"] = (
    df["staleness_score"] +
    df["visibility_score"]
)

# ------------------------------------------------------------
# ONE REASON CODE
# ------------------------------------------------------------

df["reason_code"] = np.where(
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500),
    "stale_visible_page",
    "other"
)

# ------------------------------------------------------------
# ONE ACTION LABEL
# ------------------------------------------------------------

df["action"] = np.where(
    df["reason_code"] == "stale_visible_page",
    "REVIEW_REFRESH",
    "MONITOR"
)

# ------------------------------------------------------------
# RANK
# ------------------------------------------------------------

df = df.sort_values(
    [
        "baseline_score",
        "impressions_90d",
        "days_since_last_update"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

output_columns = [
    "rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "impressions_90d"
]

queue = df[output_columns].copy()

queue.to_csv(
    output_path,
    index=False
)

print("=" * 60)
print("BASELINE QUEUE CREATED")
print("=" * 60)

print(f"Rows ranked: {len(queue):,}")
print(f"Output: {output_path}")

print("\nTop 10:")
display(queue.head(10))


BASELINE QUEUE CREATED
Rows ranked: 30,000
Output: work/outputs/baseline_action_score.csv

Top 10:


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,194,61678
1,2,content_7368877ea310,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,194,59472
2,3,content_1bfaa38ff26c,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,194,25715
3,4,content_0a91db491d14,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,193,13299
4,5,content_5feee3994adb,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,194,7812
5,6,content_c2d929d83eaa,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,193,7558
6,7,content_b16bd7307b39,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,194,4590
7,8,content_fe16a55cd13d,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,194,4556
8,9,content_ecb6215e79fd,client_7f2253d7e2,5,stale_visible_page,REVIEW_REFRESH,194,4429
9,10,content_5fe46e04994d,client_4e07408562,4,other,MONITOR,104,517715


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# I review the top 20 ranked pages individually rather than assuming
# that a high baseline score means the recommendation is correct.
# For each page, I record the action, reason code, and confidence note.
# I also identify what could make the recommendation wrong.
# Possible explanations include seasonality, stable evergreen demand,
# or insufficient search opportunity.
# The purpose of this review is to test whether the simple rule makes
# sensible recommendations when inspected by a human.

# ============================================================
# ML-07 — PART 3
# TOP-20 REVIEW
# ============================================================

import pandas as pd
import os

output_path = "work/outputs/baseline_action_score.csv"

if not os.path.exists(output_path):
    raise FileNotFoundError(
        "baseline_action_score.csv does not exist. "
        "Run Part 2 first."
    )

queue = pd.read_csv(output_path)

top20 = queue.head(20).copy()

# ------------------------------------------------------------
# CONFIDENCE NOTE
# ------------------------------------------------------------

def confidence_note(row):
    if (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 3000
    ):
        return "Strong baseline fit: very stale and highly visible."
    elif (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        return "Good baseline fit: stale and meaningfully visible."
    elif row["days_since_last_update"] >= 180:
        return "Stale, but visibility is weaker than the strongest picks."
    elif row["impressions_90d"] >= 500:
        return "Visible, but not sufficiently stale for the strongest refresh case."
    else:
        return "Weak baseline fit."

# ------------------------------------------------------------
# WHAT COULD MAKE IT WRONG
# ------------------------------------------------------------

def wrong_reason(row):
    if row["impressions_90d"] >= 3000:
        return (
            "High traffic may reflect stable demand or seasonality, "
            "so age alone does not prove a refresh is needed."
        )
    elif row["impressions_90d"] >= 500:
        return (
            "The page may still be performing adequately despite its age, "
            "or the traffic may be seasonal."
        )
    else:
        return (
            "Low visibility may mean the refresh opportunity is too small "
            "to justify prioritization."
        )

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

# ------------------------------------------------------------
# DISPLAY THE 20 REVIEWED PICKS
# ------------------------------------------------------------

review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

print("=" * 60)
print("TOP-20 REVIEW")
print("=" * 60)

display(review)

# ------------------------------------------------------------
# PLAIN-WORD SUMMARY
# ------------------------------------------------------------

print("""
TOP-20 REVIEW NOTE

Each top-ranked page is prioritized because the baseline sees a
combination of staleness and search visibility. The ranking is a
decision-support queue, not a guarantee that refreshing the page
will improve performance.

Potential explanations such as seasonality, stable evergreen demand,
or insufficient opportunity could make an individual recommendation
wrong.
""")


TOP-20 REVIEW


,rank,content_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
1,2,content_7368877ea310,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
2,3,content_1bfaa38ff26c,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
3,4,content_0a91db491d14,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
4,5,content_5feee3994adb,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
5,6,content_c2d929d83eaa,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
6,7,content_b16bd7307b39,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
7,8,content_fe16a55cd13d,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
8,9,content_ecb6215e79fd,REVIEW_REFRESH,stale_visible_page,5,Strong baseline fit: very stale and highly vis...,High traffic may reflect stable demand or seas...
9,10,content_5fe46e04994d,MONITOR,other,4,"Visible, but not sufficiently stale for the st...",High traffic may reflect stable demand or seas...



TOP-20 REVIEW NOTE

Each top-ranked page is prioritized because the baseline sees a
combination of staleness and search visibility. The ranking is a
decision-support queue, not a guarantee that refreshing the page
will improve performance.

Potential explanations such as seasonality, stable evergreen demand,
or insufficient opportunity could make an individual recommendation
wrong.



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# Weak picks are treated as possible false positives rather than
# automatic refresh recommendations.
# I check whether any weak recommendation appears inconsistent with
# the intended refresh-review rule.
# I also check that the baseline does not use label-derived fields,
# FlyRank product decision flags, or future-window information.
# This keeps the baseline leakage-safe and makes it a fair reference
# for the Week-5 model to beat.

# ============================================================
# ML-07 — PART 4
# WEAK PICKS + LEAKAGE CHECK
# ============================================================

import pandas as pd
import numpy as np
import os

output_path = "work/outputs/baseline_action_score.csv"

if not os.path.exists(output_path):
    raise FileNotFoundError(
        "baseline_action_score.csv does not exist. Run Part 2 first."
    )

queue = pd.read_csv(output_path)

# ------------------------------------------------------------
# IDENTIFY WEAK PICKS
# ------------------------------------------------------------

weak_picks = queue[
    (queue["baseline_score"] <= 2)
].head(10)

print("=" * 60)
print("WEAK PICKS")
print("=" * 60)

display(weak_picks)

print("""
Weak picks are pages that receive a relatively low baseline score
or are promoted without having both strong staleness and meaningful
visibility. These are potential false positives and should receive
human review rather than being treated as guaranteed refresh targets.
""")

# ------------------------------------------------------------
# LEAKAGE CHECK
# ------------------------------------------------------------

print("=" * 60)
print("LEAKAGE CHECK")
print("=" * 60)

# These fields must NOT be used by the baseline.
forbidden_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier"
]

used_fields = [
    "impressions_90d",
    "days_since_last_update"
]

leaked = [
    field for field in used_fields
    if field in forbidden_fields
]

if leaked:
    raise AssertionError(
        f"Leakage detected in baseline features: {leaked}"
    )

print("✓ No label-derived fields were used.")
print("✓ No FlyRank product decision flags were used.")
print("✓ No future-window fields were used.")
print("✓ Baseline uses only staleness and visibility signals.")

print("""
FINAL LEAKAGE CONCLUSION:

The baseline score is based only on observable staleness and
search-visibility measurements. It does not use trend_direction,
trend_pct, the decline label, FlyRank product flags, or future-window
outcomes. Therefore, the rule is designed as a leakage-safe
decision-support baseline.
""")


WEAK PICKS


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
11352,11353,content_5be41dbf015b,client_19581e27de,2,other,MONITOR,22,2999
11353,11354,content_6254768956a9,client_19581e27de,2,other,MONITOR,22,2996
11354,11355,content_d9f6da9d0fef,client_3fdba35f04,2,other,MONITOR,14,2995
11355,11356,content_d2d4f3955d5c,client_f74efabef1,2,other,MONITOR,20,2990
11356,11357,content_d9dc9bee7d69,client_f74efabef1,2,other,MONITOR,20,2988
11357,11358,content_039e7310835d,client_f74efabef1,2,other,MONITOR,20,2988
11358,11359,content_973ec83cacf8,client_4e07408562,2,other,MONITOR,13,2987
11359,11360,content_b1012a045720,client_f369cb89fc,2,other,MONITOR,8,2986
11360,11361,content_ebbed02286c3,client_6208ef0f77,2,other,MONITOR,20,2985
11361,11362,content_406db3266689,client_4e07408562,2,other,MONITOR,7,2983



Weak picks are pages that receive a relatively low baseline score
or are promoted without having both strong staleness and meaningful
visibility. These are potential false positives and should receive
human review rather than being treated as guaranteed refresh targets.

LEAKAGE CHECK
✓ No label-derived fields were used.
✓ No FlyRank product decision flags were used.
✓ No future-window fields were used.
✓ Baseline uses only staleness and visibility signals.

FINAL LEAKAGE CONCLUSION:

The baseline score is based only on observable staleness and
search-visibility measurements. It does not use trend_direction,
trend_pct, the decline label, FlyRank product flags, or future-window
outcomes. Therefore, the rule is designed as a leakage-safe
decision-support baseline.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.